In [ ]:
import sys
import os
from pathlib import Path  # noqa: F401

import numpy as np
import seaborn as sns

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from src.analysis.isc import (  # noqa: E402
    FREQUENCY_BANDS,
    compute_loo_isc,
    compute_loo_isc_spearman,
    compute_pairwise_isc,
    compute_pairwise_isc_spearman,
    compute_sliding_window_isc,
    compute_sliding_window_isc_spearman,
)
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)
from src.visualization.isc_plots import (  # noqa: E402
    plot_band_loo_isc_pearson_vs_spearman,
    plot_band_multiscale_sliding_window_isc,
    plot_band_overlap,
    plot_band_pairwise_isc_pearson_vs_spearman,
    print_data_overview,
)
from scripts.analysis_common import (  # noqa: E402
    BAND_ISC_THRESHOLDS,
    analyzers_to_datasets,
    load_analyzers,
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)

# EEG Inter-Subject Correlation Analysis — Per Frequency Band

This notebook computes **inter-subject correlation (ISC)** on the raw EEG
signal band-pass filtered to the five standard EEG frequency bands
(delta, theta, alpha, beta, gamma):

1. **Per-band LOO-ISC distributions** — for each band, Pearson and Spearman
   correlation against the mean of all others; histogram and per-subject violin plot
2. **Per-band pairwise ISC matrix** — symmetric subject × subject correlation
   matrix per band; per-subject mean off-diagonal for outlier detection
3. **Per-band sliding-window ISC** — time-resolved LOO-ISC per band at three
   temporal scales; Pearson vs Spearman comparison at medium window
4. **Band-overlap analysis** — raster showing which bands simultaneously
   exceed their significance thresholds

Computation functions are imported from `src.analysis.isc`.

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL, MusicTypeVariants.PSYTRANCE]
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]

# ── Data processing flag ───────────────────────────────────────────────────
process_and_save_data = False

# ── LOO-ISC distribution parameters ───────────────────────────────────────
PLOT_PCT = 99  # clip histogram at this percentile to suppress outliers

# ── Sliding-window ISC parameters ─────────────────────────────────────────
# Three window sizes with 50% overlap (step = window / 2), same as broadband analysis.
WINDOW_FINE_SEC = 1.0  # fine / one-step window (seconds)
WINDOW_MED_SEC = 5.0  # medium window (seconds)
WINDOW_LARGE_SEC = 15.0  # large window (seconds)

# Per-band ISC significance thresholds
BAND_THRESHOLDS = BAND_ISC_THRESHOLDS

# Sub-sample channels for Spearman to keep runtime manageable during exploration.
# Set to None to use all channels (much slower).
N_CH_SUBSAMPLE = 64

# ── Plot saving ────────────────────────────────────────────────────────────
SAVE_PLOTS = True

PLOTS_DIR = ProjectPaths.NOTEBOOKS_DIR / "02-isc-broadband-analysis" / "plots" / "bands"
print(f"Plots will be saved to: {PLOTS_DIR}")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

## Data Loading

In [ ]:
# Load (or process-and-save) one EEGSummarizedAnalyzer per music type.
# normalize_data=False: keep raw amplitudes for this ISC analysis.
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    process_and_save_data,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)
print_data_overview(datasets)

## Dataset Selection

Change `LABEL` to switch between music types. All analysis cells below use
`ad`, `data`, `n_subjects`, `n_channels`, `n_times`, and `sfreq`.

In [ ]:
LABEL = MusicTypeVariants.CLASSICAL.value
# LABEL = MusicTypeVariants.PSYTRANCE.value

ad = datasets[f"{CONDITION.value}_{LABEL}"]
data = ad.data  # (n_subjects, n_channels, n_times) — unnormalized
sfreq = ad.sfreq

n_subjects, n_channels, n_times = data.shape
time = np.arange(n_times) / sfreq  # seconds

print(f"Dataset : {LABEL}")
print(f"Shape   : {data.shape}  (subjects × channels × time points)")
print(f"Duration: {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print("Data: unnormalized (raw amplitude).")

# Available frequency bands
print("\nFrequency bands:")
for band, (lo, hi) in FREQUENCY_BANDS.items():
    thr = BAND_THRESHOLDS.get(band, 0.035)
    print(f"  {band:6s}: {lo}–{hi} Hz  (threshold r = {thr})")

## Section 1 — Per-Band LOO-ISC Distribution (Pearson vs. Spearman)

For each frequency band, the raw EEG is band-pass filtered and leave-one-out
ISC is computed per channel for each subject.

For each band:
- **Histogram** — distribution of channel-wise mean LOO-ISC (Pearson vs. Spearman)
- **Per-subject violin + strip** — each dot is one subject's mean band-LOO-ISC
  averaged over channels; reveals band-specific outlier subjects

In [ ]:
# Channel subsampling for Spearman (same seed as broadband for reproducibility)
if N_CH_SUBSAMPLE is not None and N_CH_SUBSAMPLE < n_channels:
    rng = np.random.default_rng(42)
    ch_idx = np.sort(rng.choice(n_channels, N_CH_SUBSAMPLE, replace=False))
    print(
        f"Using {N_CH_SUBSAMPLE}/{n_channels} randomly subsampled channels for Spearman."
    )
else:
    ch_idx = np.arange(n_channels)

band_iscs: dict[str, tuple] = {}  # (loo_pearson, mean_pearson)
band_iscs_spearman: dict[
    str, tuple
] = {}  # (loo_spearman, mean_spearman) on subsampled channels

for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    print(f"  {band:6s} ({l_freq}\u2013{h_freq} Hz) \u2014 Pearson LOO-ISC \u2026")
    filtered = ad.filter_to_band(l_freq, h_freq)
    loo, mean_isc = compute_loo_isc(filtered.data)
    band_iscs[band] = (loo, mean_isc)

    print(f"  {band:6s} ({l_freq}\u2013{h_freq} Hz) \u2014 Spearman LOO-ISC \u2026")
    loo_sp, mean_isc_sp = compute_loo_isc_spearman(filtered.data[:, ch_idx, :])
    band_iscs_spearman[band] = (loo_sp, mean_isc_sp)

plot_band_loo_isc_pearson_vs_spearman(
    LABEL,
    band_iscs,
    band_iscs_spearman,
    plot_pct=PLOT_PCT,
    save_path_dir=PLOTS_DIR if SAVE_PLOTS else None,
)

## Section 2 — Per-Band Pairwise ISC Matrix (Pearson vs. Spearman)

For each frequency band, every pair of subjects is correlated (mean over channels):

- **Heatmaps** — diagonal masked so the colour scale focuses on off-diagonal spread
- **Per-subject mean off-diagonal bar chart** — subjects with consistently low values
  are candidates for quality review
- **Off-diagonal distribution** — histogram comparing Pearson vs. Spearman spread

In [ ]:
band_pair_pearson: dict[str, np.ndarray] = {}
band_pair_spearman: dict[str, np.ndarray] = {}

for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    print(f"  {band:6s} \u2014 Pearson pairwise ISC \u2026")
    filtered = ad.filter_to_band(l_freq, h_freq)
    band_pair_pearson[band] = compute_pairwise_isc(filtered.data)

    print(f"  {band:6s} \u2014 Spearman pairwise ISC \u2026")
    band_pair_spearman[band] = compute_pairwise_isc_spearman(
        filtered.data[:, ch_idx, :]
    )

plot_band_pairwise_isc_pearson_vs_spearman(
    LABEL,
    band_pair_pearson,
    band_pair_spearman,
    save_path_dir=PLOTS_DIR if SAVE_PLOTS else None,
)

## Section 3 — Per-Band Sliding-Window ISC (Multi-Scale)

Time-resolved LOO-ISC per frequency band at three temporal scales (50% overlap, step = window / 2).

For each band:
- **Figure A** — Bar chart (medium window, Pearson): per-window mean ISC
- **Figure B** — 3-scale stair-step overlay + per-channel heatmap (Pearson)
- **Figure C** — Pearson vs. Spearman comparison at medium window

In [ ]:
# Channel subsampling (same subset as Section 1)
data_sw = data[:, ch_idx, :]

# ── Compute LOO-ISC at 3 window sizes for each band ─────────────────────────
band_sw_fine: dict[str, tuple] = {}
band_sw_med: dict[str, tuple] = {}
band_sw_large: dict[str, tuple] = {}
band_sw_spearman_med: dict[str, tuple] = {}

for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    filtered_sw = ad.filter_to_band(l_freq, h_freq)
    fdata_sw = filtered_sw.data[:, ch_idx, :]

    print(f"  {band:6s} Pearson ISC ({WINDOW_FINE_SEC:.0f} s / 50% overlap) \u2026")
    isc_tc, sw_times = compute_sliding_window_isc(
        fdata_sw, WINDOW_FINE_SEC, WINDOW_FINE_SEC / 2, sfreq
    )
    band_sw_fine[band] = (isc_tc, sw_times)

    print(f"  {band:6s} Pearson ISC ({WINDOW_MED_SEC:.0f} s / 50% overlap) \u2026")
    isc_tc, sw_times = compute_sliding_window_isc(
        fdata_sw, WINDOW_MED_SEC, WINDOW_MED_SEC / 2, sfreq
    )
    band_sw_med[band] = (isc_tc, sw_times)

    print(f"  {band:6s} Pearson ISC ({WINDOW_LARGE_SEC:.0f} s / 50% overlap) \u2026")
    isc_tc, sw_times = compute_sliding_window_isc(
        fdata_sw, WINDOW_LARGE_SEC, WINDOW_LARGE_SEC / 2, sfreq
    )
    band_sw_large[band] = (isc_tc, sw_times)

    print(f"  {band:6s} Spearman ISC ({WINDOW_MED_SEC:.0f} s / 50% overlap) \u2026")
    isc_sp, sw_sp_times = compute_sliding_window_isc_spearman(
        fdata_sw, WINDOW_MED_SEC, WINDOW_MED_SEC / 2, sfreq
    )
    band_sw_spearman_med[band] = (isc_sp, sw_sp_times)

plot_band_multiscale_sliding_window_isc(
    LABEL,
    band_sw_fine,
    band_sw_med,
    band_sw_large,
    band_sw_spearman_med,
    sfreq,
    n_times,
    window_fine_sec=WINDOW_FINE_SEC,
    window_med_sec=WINDOW_MED_SEC,
    window_large_sec=WINDOW_LARGE_SEC,
    band_thresholds=BAND_THRESHOLDS,
    n_ch_subsample=N_CH_SUBSAMPLE
    if N_CH_SUBSAMPLE is not None and N_CH_SUBSAMPLE < n_channels
    else None,
    save_path_dir=PLOTS_DIR if SAVE_PLOTS else None,
)

## Section 4 — Band-Overlap Analysis

A raster plot showing, for each time window, which frequency bands simultaneously
exceed their respective ISC significance thresholds.

Windows where multiple bands show significant synchrony may indicate periods of
especially strong neural alignment across participants.

In [ ]:
# Broadband sliding-window ISC (required for the overlap raster)
print("Computing broadband sliding-window ISC for band-overlap raster …")
sw_isc_bb, sw_times_bb = compute_sliding_window_isc(
    data,
    window_sec=WINDOW_MED_SEC,
    step_sec=WINDOW_MED_SEC / 2,
    sfreq=sfreq,
)

# Build band_sw dict using medium-window sliding-window results
band_sw: dict[str, tuple] = {}
for band, (l_freq, h_freq) in FREQUENCY_BANDS.items():
    filtered = ad.filter_to_band(l_freq, h_freq)
    tc, times = compute_sliding_window_isc(
        filtered.data,
        window_sec=WINDOW_MED_SEC,
        step_sec=WINDOW_MED_SEC / 2,
        sfreq=sfreq,
    )
    band_sw[band] = (tc, times)

fig_overlap = plot_band_overlap(
    {LABEL: band_sw},
    bands=FREQUENCY_BANDS,
    band_thresholds=BAND_THRESHOLDS,
    broadband_sw={LABEL: (sw_isc_bb, sw_times_bb)},
    broadband_threshold=0.035,
    save_path=PLOTS_DIR / f"band_overlap_{LABEL}.png" if SAVE_PLOTS else None,
)